# Esercizio 3.2 - Adattamento parameter-efficient di CLIP

Kernel consigliato: `clip_lora`.

Obiettivo: valutare CLIP zero-shot su ImageNet-Sketch e poi adattarlo con un metodo parameter-efficient. Usiamo CLIP-Adapter sulle feature immagine perche' e' leggero, chiaro e adatto a GPU con poca VRAM.

Nota: questo notebook usa un adapter, non una LoRA interna al transformer. E' comunque parameter-efficient: il backbone CLIP resta congelato e si addestra solo un piccolo MLP.


> **Nota di esecuzione**
>
> Gli output visibili sono stati prodotti durante le esecuzioni finali o di validazione del laboratorio. Nella versione di consegna i training costosi sono disattivati di default quando sono controllati da flag; checkpoint e artefatti salvati vengono usati per consultazione rapida.


In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

PROJECT_ROOT


Path('DLA_2')

In [2]:
from torch.utils.data import DataLoader
import pandas as pd
import torch

from dla_lab2.clip_utils import (
    CLIPAdapter,
    build_clip_tensor_dataset,
    build_text_features,
    build_text_features_ensemble,
    evaluate_adapter,
    evaluate_precomputed_features,
    load_imagenet_labels,
    load_imagenet_sketch,
    load_open_clip_model,
    precompute_image_features,
    split_tensor_dataset,
    train_clip_adapter,
)
from dla_lab2.seed import set_seed

set_seed(42)

# Download, preprocessing e training CLIP sono disattivati nella modalita' rapida.
RUN_CLIP_EXPERIMENT = False


## 1. Modello e labels

Carichiamo CLIP ViT-B/16 e i nomi semplici delle 1000 classi ImageNet. Usiamo la variante quickgelu per coerenza con i pesi OpenAI.


In [3]:
if RUN_CLIP_EXPERIMENT:
    model, preprocess, tokenizer, device = load_open_clip_model(
        model_name="ViT-B-16-quickgelu",
        pretrained="openai",
    )

    imagenet_class_names = load_imagenet_labels(PROJECT_ROOT / "imagenet_labels.json")
    print(device)
    print(len(imagenet_class_names), imagenet_class_names[:5])
else:
    print("Modello CLIP non caricato.")


cuda
1000 ['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead shark']


## 2. Dataset ImageNet-Sketch

ImageNet-Sketch e' piu' interessante di ImageNette perche' introduce domain shift: CLIP vede disegni e schizzi invece di foto naturali.


In [4]:
if RUN_CLIP_EXPERIMENT:
    sketch_train, sketch_val = load_imagenet_sketch(seed=42, train_fraction=0.8)
    print(len(sketch_train), len(sketch_val))
else:
    print("Dataset ImageNet-Sketch non caricato.")


Dataset ImageNet-Sketch non caricato.


In [5]:
if RUN_CLIP_EXPERIMENT:
    TRAIN_SAMPLES = 5000
    BATCH_SIZE = 64

    # Usiamo un sottoinsieme del training split per mantenere il laboratorio gestibile.
    # La validation esterna resta completa, cosi' la valutazione finale e' piu' stabile.
    train_tensor_ds = build_clip_tensor_dataset(sketch_train, preprocess, num_samples=TRAIN_SAMPLES)
    val_tensor_ds = build_clip_tensor_dataset(sketch_val, preprocess, num_samples=None)

    train_image_loader = DataLoader(train_tensor_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    val_image_loader = DataLoader(val_tensor_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print(f"Full train split examples: {len(sketch_train)}")
    print(f"Train subset examples used: {len(train_tensor_ds)}")
    print(f"External validation examples used: {len(val_tensor_ds)}")
    print(f"Image batch size: {BATCH_SIZE}")
else:
    print("Preprocessing immagini non eseguito.")


Preprocessing images:   0%|          | 0/5000 [00:00<?, ?it/s]

Preprocessing images:   0%|          | 0/10178 [00:00<?, ?it/s]

Full train split examples: 40711
Train subset examples used: 5000
External validation examples used: 10178
Image batch size: 64


## 3. Zero-shot baseline e prompt study

Prima valutiamo CLIP senza training e confrontiamo alcuni prompt semplici. CLIP e' sensibile al testo usato come descrizione della classe: cambiare prompt puo' cambiare leggermente l'accuracy.

Il dataset e' stato diviso inizialmente in modo standard `80/20`: `40.711` esempi nello split train e `10.178` nello split validation. Per il training dell'adapter usiamo solo `5.000` esempi del train split, scelta dichiarata e riproducibile. La validation esterna rimane completa per avere una stima piu' affidabile del risultato finale.

Buone pratiche computazionale: il visual encoder di CLIP resta congelato, quindi calcoliamo le feature immagine del validation set una sola volta e poi riusiamo quelle feature per tutti i prompt.


### Similarità CLIP e scala dei logit

CLIP confronta embedding immagine e testo normalizzati:

$$
\mathrm{cos}\!\left(f_I(x),f_T(t)\right)
=
\frac{f_I(x)^\top f_T(t)}
{\lVert f_I(x)\rVert_2\lVert f_T(t)\rVert_2}
$$

dove:

- $x$ è un'immagine e $t$ è un prompt testuale di classe;
- $f_I(x)$ è l'embedding prodotto dall'encoder di immagini;
- $f_T(t)$ è l'embedding prodotto dall'encoder di testo;
- il prodotto scalare al numeratore è normalizzato dalle due norme euclidee.

Valori più alti indicano maggiore compatibilità tra immagine e testo. `encode_text_prompts` e `precompute_image_features` normalizzano gli embedding; `evaluate_zero_shot` ed `evaluate_precomputed_features` calcolano il prodotto matriciale e applicano poi la scala dei logit fornita da `model.logit_scale.exp()`. Il confronto nello spazio condiviso permette la classificazione zero-shot senza una testa supervisionata ImageNet-Sketch.


In [6]:
if RUN_CLIP_EXPERIMENT:
    prompt_templates = [
        "a sketch of a {}",
        "a black and white sketch of a {}",
        "a hand-drawn sketch of a {}",
        "a pencil drawing of a {}",
        "a line drawing of a {}",
    ]

    logit_scale = model.logit_scale.exp().item()

    # Calcoliamo le feature del validation set una sola volta.
    # Questo rende il prompt study molto piu' veloce rispetto a rieseguire CLIP per ogni prompt.
    val_feature_ds = precompute_image_features(model, val_image_loader, device)
    val_feature_loader = DataLoader(val_feature_ds, batch_size=256, shuffle=False)

    prompt_results = []
    for template in prompt_templates:
        text_features = build_text_features(model, tokenizer, imagenet_class_names, template, device)
        acc = evaluate_precomputed_features(val_feature_loader, text_features, logit_scale, device)
        prompt_results.append({"prompt": template, "accuracy": acc})

    prompt_table = pd.DataFrame(prompt_results).sort_values("accuracy", ascending=False)
    prompt_table
else:
    print("Prompt study non rieseguito.")


,prompt,accuracy
2,a hand-drawn sketch of a {},0.475928
4,a line drawing of a {},0.473570
1,a black and white sketch of a {},0.472195
0,a sketch of a {},0.465612
3,a pencil drawing of a {},0.462173


Osservazioni dalla run zero-shot:

- miglior prompt singolo: `a hand-drawn sketch of a {}` con accuracy `0.4759`;
- il prompt peggiore tra quelli provati resta vicino (`0.4622`), quindi la scelta del prompt conta ma non cambia completamente il risultato;
- il prompt study serve a scegliere una formulazione testuale ragionevole prima di addestrare l'adapter.

Con `prompt ensemble` intendiamo la media delle feature testuali prodotte da piu' prompt per la stessa classe. Invece di usare una sola frase, combiniamo piu' descrizioni e usiamo la rappresentazione media come prototipo testuale della classe.


## 4. CLIP-Adapter

Congeliamo CLIP e addestriamo solo un adapter MLP sulle feature immagine gia' precalcolate. Questo riduce memoria e rischio di danneggiare le rappresentazioni zero-shot.

Il `bottleneck` e' la dimensione del livello nascosto dell'adapter. Un bottleneck piu' piccolo usa meno parametri e forza una correzione piu' semplice; un bottleneck piu' grande e' piu' flessibile ma puo' overfittare piu' facilmente.

Parametri addestrabili dell'adapter:

- bottleneck 64: `66.113` parametri;
- bottleneck 128: `131.713` parametri.

Rispetto al backbone CLIP completo sono una frazione molto piccola: per questo il metodo e' parameter-efficient.


### Adapter residuale con bottleneck

L'adapter implementa

$$
h'=h+\sigma(\alpha)\,W_{\mathrm{up}}\mathrm{ReLU}(W_{\mathrm{down}}h).
$$

dove:

- $h$ è la feature immagine prodotta da CLIP e $h'$ quella adattata;
- $W_{\mathrm{down}}$ riduce la dimensione nel bottleneck;
- $W_{\mathrm{up}}$ ripristina la dimensione originale;
- $\mathrm{ReLU}$ introduce la non linearità;
- $\sigma(\alpha)$ è il gate sigmoid addestrabile del ramo residuo.

Il collegamento residuo conserva la feature originale e aggiunge una correzione leggera. `CLIPAdapter.forward` implementa esattamente questa struttura; il risultato migliore usa bottleneck 128 senza aggiornare gli encoder CLIP. Prima della cross-entropy, `train_clip_adapter` normalizza nuovamente $h'$ e usa gli stessi logit CLIP scalati.


In [7]:
if RUN_CLIP_EXPERIMENT:
    train_feature_ds = precompute_image_features(model, train_image_loader, device)
    adapter_train_ds, adapter_val_ds = split_tensor_dataset(train_feature_ds, train_fraction=0.9, seed=42)

    adapter_train_loader = DataLoader(adapter_train_ds, batch_size=256, shuffle=True)
    adapter_val_loader = DataLoader(adapter_val_ds, batch_size=256, shuffle=False)

    print(f"Train feature examples: {len(train_feature_ds)}")
    print(f"Adapter train/validation split: {len(adapter_train_ds)} / {len(adapter_val_ds)}")
    print(f"External validation feature examples: {len(val_feature_ds)}")
else:
    print("Feature CLIP di training non ricalcolate.")


Feature CLIP di training non ricalcolate.


In [8]:
if RUN_CLIP_EXPERIMENT:
    base_prompt = prompt_table.iloc[0]["prompt"]
    text_features_base = build_text_features(model, tokenizer, imagenet_class_names, base_prompt, device)

    adapter64 = CLIPAdapter(feat_dim=512, bottleneck=64, alpha=0.6)
    history64 = train_clip_adapter(
        adapter64,
        adapter_train_loader,
        adapter_val_loader,
        text_features_base,
        logit_scale,
        device,
        epochs=30,
        lr=2e-3,
    )

    adapter128 = CLIPAdapter(feat_dim=512, bottleneck=128, alpha=0.6)
    history128 = train_clip_adapter(
        adapter128,
        adapter_train_loader,
        adapter_val_loader,
        text_features_base,
        logit_scale,
        device,
        epochs=30,
        lr=2e-3,
    )
else:
    print("Training degli adapter non eseguito.")


Training degli adapter non eseguito.


## 5. Valutazione finale

Valutiamo zero-shot, adapter singolo e adapter con prompt ensembling sul validation set esterno ImageNet-Sketch. Usiamo sempre le feature immagine precalcolate, quindi il confronto finale non riesegue inutilmente il visual encoder.


In [9]:
if RUN_CLIP_EXPERIMENT:
    zeroshot_acc = evaluate_precomputed_features(val_feature_loader, text_features_base, logit_scale, device)
    acc64 = evaluate_adapter(adapter64, val_feature_loader, text_features_base, logit_scale, device)
    acc128 = evaluate_adapter(adapter128, val_feature_loader, text_features_base, logit_scale, device)

    text_features_ensemble = build_text_features_ensemble(model, tokenizer, imagenet_class_names, prompt_templates, device)
    zs_ensemble = evaluate_precomputed_features(val_feature_loader, text_features_ensemble, logit_scale, device)
    acc64_ensemble = evaluate_adapter(adapter64, val_feature_loader, text_features_ensemble, logit_scale, device)
    acc128_ensemble = evaluate_adapter(adapter128, val_feature_loader, text_features_ensemble, logit_scale, device)

    results = pd.DataFrame(
        [
            {"method": "Zero-shot CLIP", "accuracy": zeroshot_acc, "gain": 0.0},
            {"method": "Zero-shot CLIP + prompt ensemble", "accuracy": zs_ensemble, "gain": zs_ensemble - zeroshot_acc},
            {"method": "CLIP-Adapter bottleneck=64", "accuracy": acc64, "gain": acc64 - zeroshot_acc},
            {"method": "CLIP-Adapter b=64 + prompt ensemble", "accuracy": acc64_ensemble, "gain": acc64_ensemble - zeroshot_acc},
            {"method": "CLIP-Adapter bottleneck=128", "accuracy": acc128, "gain": acc128 - zeroshot_acc},
            {"method": "CLIP-Adapter b=128 + prompt ensemble", "accuracy": acc128_ensemble, "gain": acc128_ensemble - zeroshot_acc},
        ]
    )
    results
else:
    results = pd.read_csv(PROJECT_ROOT / "results" / "clip_results.csv")
    display(results)


,method,accuracy,gain_vs_single_prompt_zero_shot,source
0,Zero-shot CLIP,0.475928,0.000000,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
1,Zero-shot CLIP + prompt ensemble,0.478876,0.002948,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
2,CLIP-Adapter bottleneck 64,0.514443,0.038514,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
3,CLIP-Adapter bottleneck 64 + prompt ensemble,0.506681,0.030753,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
4,CLIP-Adapter bottleneck 128,0.524072,0.048143,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
5,CLIP-Adapter bottleneck 128 + prompt ensemble,0.519257,0.043329,notebooks/04_clip_adapter_imagenet_sketch.ipyn...


In [10]:
if RUN_CLIP_EXPERIMENT:
    # Questa tabella serve solo a controllare l'andamento della loss nelle ultime epoche.
    # La scelta finale del modello si basa sulla tabella di accuracy esterna sopra.
    loss_tail = pd.concat(
        [
            pd.DataFrame(history64).assign(adapter="bottleneck=64").tail(),
            pd.DataFrame(history128).assign(adapter="bottleneck=128").tail(),
        ],
        ignore_index=True,
    )
    loss_tail
else:
    loss_tail = pd.read_csv(PROJECT_ROOT / "results" / "adapter_loss_tail.csv")
    display(loss_tail)


,adapter,epoch,train_loss,validation_loss,source
0,bottleneck 64,26,0.302842,2.151163,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
1,bottleneck 64,27,0.301724,2.150304,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
2,bottleneck 64,28,0.300533,2.151331,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
3,bottleneck 64,29,0.296222,2.151732,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
4,bottleneck 64,30,0.296206,2.151857,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
5,bottleneck 128,26,0.096574,2.182509,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
6,bottleneck 128,27,0.095409,2.182874,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
7,bottleneck 128,28,0.095256,2.182661,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
8,bottleneck 128,29,0.094519,2.183435,notebooks/04_clip_adapter_imagenet_sketch.ipyn...
9,bottleneck 128,30,0.093990,2.183509,notebooks/04_clip_adapter_imagenet_sketch.ipyn...


Osservazioni dalla valutazione finale:

- baseline zero-shot: accuracy `0.4759`;
- prompt ensemble senza training: accuracy `0.4789`, quindi migliora poco (`+0.0029`);
- CLIP-Adapter bottleneck 64: accuracy `0.5144`, miglioramento `+0.0385`;
- CLIP-Adapter bottleneck 128: accuracy `0.5241`, miglioramento `+0.0481`, migliore risultato della run;
- l'ensemble dei prompt non migliora gli adapter in questa run: il miglior risultato resta adapter bottleneck 128 con il miglior prompt singolo.

La tabella delle loss mostra che il bottleneck 128 riduce molto la train loss, ma mantiene una validation loss alta. Questo e' un segnale di possibile overfitting. Tuttavia la valutazione esterna mostra comunque un miglioramento di accuracy rispetto allo zero-shot, quindi il risultato e' utile e spiegabile.


## Conclusioni Esercizio 3.2

Tutti i punti richiesti sono stati svolti.

- Abbiamo usato un modello CLIP piccolo e standard: `ViT-B-16-quickgelu` con pesi OpenAI.
- Abbiamo scelto ImageNet-Sketch, un dataset adatto per testare domain shift perche' contiene schizzi invece di foto naturali.
- Abbiamo valutato CLIP in zero-shot con piu' prompt: il miglior prompt singolo ha ottenuto accuracy `0.4759`.
- Abbiamo applicato un metodo parameter-efficient: CLIP-Adapter sulle feature immagine, lasciando congelati image encoder e text encoder.
- Il miglior adapter, bottleneck 128, ha raggiunto accuracy `0.5241`, con gain `+0.0481` rispetto allo zero-shot.

Interpretazione: CLIP parte gia' da una baseline forte su ImageNet-Sketch, ma un piccolo adapter migliora il dominio sketch senza aggiornare tutto il modello. Il bottleneck 128 funziona meglio del 64 in accuracy esterna, anche se la loss suggerisce possibile overfitting. Il risultato e' accettabile per l'esercizio perche' il miglioramento e' misurato, confrontato con la baseline e spiegato.

Buone pratiche adottate:

- split iniziale `80/20` del dataset;
- uso dichiarato di un sottoinsieme del train split per rendere il laboratorio gestibile;
- validation esterna completa per il confronto finale;
- backbone CLIP congelato;
- feature immagine precalcolate;
- confronto con zero-shot, prompt singolo e prompt ensemble;
- reporting del gain rispetto alla baseline;
- scelta finale basata su accuracy esterna, non solo su train loss.


## Funzioni, classi e moduli locali richiamati

La tabella è ricavata dagli import, dagli utilizzi nelle celle e dalle definizioni effettive nei package locali.

| Funzione o classe | Tipo | File di definizione | Scopo | Input principali | Output principali | Sezione |
| ----------------- | ---- | ------------------- | ----- | ---------------- | ----------------- | ------- |
| `build_clip_tensor_dataset` | Funzione | `src/dla_lab2/clip_utils.py` | Preprocessa immagini HF in tensori CLIP e conserva le label. | `hf_dataset: Any`, `preprocess: Any`, `num_samples: int \| None` | TensorDataset `(pixel_values, labels)`. | 2. Dataset ImageNet-Sketch |
| `build_text_features` | Funzione | `src/dla_lab2/clip_utils.py` | Costruisce le feature testuali normalizzate per tutte le classi. | `model: Any`, `tokenizer: Any`, `class_names: list[str]`, `prompt_template: str`, `device: str`, `batch_size: int` | Tensore `(n_classi, feature_dim)` normalizzato. | 3. Zero-shot baseline e prompt study; 4. CLIP-Adapter |
| `build_text_features_ensemble` | Funzione | `src/dla_lab2/clip_utils.py` | Media feature testuali ottenute da piu' prompt. | `model: Any`, `tokenizer: Any`, `class_names: list[str]`, `prompt_templates: list[str]`, `device: str` | Tensore normalizzato `(n_classi, feature_dim)`. | 5. Valutazione finale |
| `CLIPAdapter` | Classe | `src/dla_lab2/clip_utils.py` | Adapter leggero applicato alle feature immagine di CLIP. | `feat_dim: int`, `bottleneck: int`, `alpha: float` | Istanza configurata della classe. | 4. CLIP-Adapter |
| `evaluate_adapter` | Funzione | `src/dla_lab2/clip_utils.py` | Valuta un adapter CLIP su feature immagine precalcolate. | `adapter: CLIPAdapter`, `loader: DataLoader`, `text_features: torch.Tensor`, `logit_scale: float`, `device: str` | Accuracy come float tra 0 e 1. | 5. Valutazione finale |
| `evaluate_precomputed_features` | Funzione | `src/dla_lab2/clip_utils.py` | Valuta una classificazione zero-shot usando feature immagine gia' calcolate. | `loader: DataLoader`, `text_features: torch.Tensor`, `logit_scale: float`, `device: str` | Accuracy come float tra 0 e 1. | 3. Zero-shot baseline e prompt study; 5. Valutazione finale |
| `load_imagenet_labels` | Funzione | `src/dla_lab2/clip_utils.py` | Carica i nomi semplici delle 1000 classi ImageNet. | `path: str \| Path`, `download_if_missing: bool` | Lista di 1000 nomi classe. | 1. Modello e labels |
| `load_imagenet_sketch` | Funzione | `src/dla_lab2/clip_utils.py` | Carica ImageNet-Sketch e crea split train/validation deterministici. | `seed: int`, `train_fraction: float` | Coppia `(sketch_train, sketch_val)` di Dataset Hugging Face. | 2. Dataset ImageNet-Sketch |
| `load_open_clip_model` | Funzione | `src/dla_lab2/clip_utils.py` | Carica un modello CLIP da open_clip. | `model_name: str`, `pretrained: str`, `device: str \| None` | Tupla `(model, preprocess, tokenizer, device)`. | 1. Modello e labels |
| `precompute_image_features` | Funzione | `src/dla_lab2/clip_utils.py` | Precalcola le feature immagine CLIP per addestrare adapter leggeri. | `model: Any`, `loader: DataLoader`, `device: str` | TensorDataset `(features, labels)` su CPU. | 3. Zero-shot baseline e prompt study; 4. CLIP-Adapter |
| `set_seed` | Funzione | `src/dla_lab2/seed.py` | Imposta i seed principali per rendere gli esperimenti piu' riproducibili. | `seed: int` | None. La funzione modifica lo stato globale dei generatori casuali. | Esercizio 3.2 - Adattamento parameter-efficient di CLIP |
| `split_tensor_dataset` | Funzione | `src/dla_lab2/clip_utils.py` | Divide un TensorDataset in train e validation. | `dataset: TensorDataset`, `train_fraction: float`, `seed: int` | Coppia `(train_dataset, val_dataset)`. | 4. CLIP-Adapter |
| `train_clip_adapter` | Funzione | `src/dla_lab2/clip_utils.py` | Addestra un CLIPAdapter su feature immagine gia' precalcolate. | `adapter: CLIPAdapter`, `train_loader: DataLoader`, `val_loader: DataLoader`, `text_features: torch.Tensor`, `logit_scale: float`, `device: str`, `epochs: int`, `lr: float` | Lista di dizionari con loss train e validation per epoca. | 4. CLIP-Adapter |
